# 02 — Amenity Composition

Queries OSM via Overpass API for amenities within each census tract and computes per-tract counts, ratios, and density.

**Data source:** Overpass API (OSM) — fully portable to any city.

**Method:**
1. Load tract centroids from `csv/01_zone_definition.csv`
2. For each tract, query Overpass for amenities within a 500m radius of centroid
3. Categorize into: food_drink, retail, services, office, hotel, entertainment, education, healthcare
4. Compute raw counts, ratios, and density per tract

**Output columns:** `tract_id`, `amenity_count_total`, `amenity_count_food`, `amenity_count_retail`, `amenity_count_services`, `amenity_count_office`, `amenity_count_hotel`, `amenity_count_entertainment`, `amenity_count_education`, `amenity_count_healthcare`, `amenity_ratio_food`, `amenity_ratio_retail`, `amenity_density`

**Output file:** `csv/02_amenity_composition.csv`

In [ ]:
# ── Papermill parameters ──────────────────────────────
ZONES_CONFIG = "zones.json"
QUERY_RADIUS = 500   # meters around each tract centroid

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import json
import os
import hashlib

os.makedirs("csv", exist_ok=True)
os.makedirs("cache", exist_ok=True)

df_tracts = pd.read_csv("csv/01_zone_definition.csv", dtype={"tract_id": str})
print(f"Loaded {len(df_tracts)} tracts")

In [ ]:
# ── Overpass API configuration ────────────────────────

OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]
HEADERS = {"User-Agent": "zone-finding/1.0 (research project)"}

# Tags to query — broad set for zone characterization
QUERY_TAGS = ["amenity", "shop", "office", "craft", "tourism", "leisure"]

# ── Amenity categorization ────────────────────────────
# Maps primary_value → category for zone-level aggregation
AMENITY_CATEGORIES = {
    # food_drink
    "restaurant": "food_drink", "fast_food": "food_drink",
    "cafe": "food_drink", "bar": "food_drink", "pub": "food_drink",
    "food_court": "food_drink", "bakery": "food_drink",
    "ice_cream": "food_drink", "deli": "food_drink",
    "confectionery": "food_drink", "coffee": "food_drink",
    "tea": "food_drink", "beverages": "food_drink",
    "butcher": "food_drink", "greengrocer": "food_drink",
    "pastry": "food_drink", "brewery": "food_drink",
    "grocery": "food_drink", "seafood": "food_drink",
    # retail
    "clothes": "retail", "shoes": "retail", "jewelry": "retail",
    "gift": "retail", "convenience": "retail", "supermarket": "retail",
    "department_store": "retail", "mobile_phone": "retail",
    "electronics": "retail", "books": "retail", "furniture": "retail",
    "hardware": "retail", "cosmetics": "retail", "watches": "retail",
    "bag": "retail", "art": "retail", "antiques": "retail",
    "florist": "retail", "alcohol": "retail", "wine": "retail",
    "stationery": "retail", "toys": "retail", "pet": "retail",
    "variety_store": "retail", "marketplace": "retail",
    "fabric": "retail", "fashion_accessories": "retail",
    "perfumery": "retail", "chemist": "retail", "tobacco": "retail",
    "newsagent": "retail", "kiosk": "retail",
    # office
    "company": "office", "lawyer": "office", "accountant": "office",
    "it": "office", "consulting": "office", "estate_agent": "office",
    "financial_advisor": "office", "architect": "office",
    "tax_advisor": "office", "marketing": "office",
    "property_management": "office", "engineer": "office",
    "advertising_agency": "office", "ngo": "office",
    "coworking": "office", "coworking_space": "office",
    # hotel
    "hotel": "hotel", "hostel": "hotel", "motel": "hotel",
    "guest_house": "hotel",
    # entertainment
    "theatre": "entertainment", "cinema": "entertainment",
    "nightclub": "entertainment", "arts_centre": "entertainment",
    "gallery": "entertainment", "museum": "entertainment",
    "events_venue": "entertainment", "escape_game": "entertainment",
    "fitness_centre": "entertainment", "sports_centre": "entertainment",
    "karaoke": "entertainment", "dance": "entertainment",
    "swimming_pool": "entertainment",
    # education
    "school": "education", "university": "education",
    "college": "education", "kindergarten": "education",
    "library": "education", "language_school": "education",
    "music_school": "education", "driving_school": "education",
    # healthcare
    "pharmacy": "healthcare", "dentist": "healthcare",
    "clinic": "healthcare", "doctors": "healthcare",
    "hospital": "healthcare", "optician": "healthcare",
    "veterinary": "healthcare",
    # services (catch-all for remaining commercial)
    "hairdresser": "services", "beauty": "services",
    "massage": "services", "tattoo": "services",
    "laundry": "services", "dry_cleaning": "services",
    "bank": "services", "atm": "services",
    "bureau_de_change": "services", "travel_agency": "services",
    "photo": "services", "copyshop": "services",
    "tailor": "services", "shoemaker": "services",
}

CATEGORIES = ["food_drink", "retail", "services", "office", "hotel",
              "entertainment", "education", "healthcare"]

print(f"Amenity categories: {len(CATEGORIES)}")
print(f"Mapped values: {len(AMENITY_CATEGORIES)}")

In [ ]:
# ── Overpass query and caching ────────────────────────

def _cache_path(query):
    h = hashlib.sha1(query.encode()).hexdigest()
    return f"cache/{h}.json"


def query_overpass(lat, lon, radius, max_retries=3):
    """Query Overpass API for all tagged features near a point."""
    tag_filters = "\n    ".join(
        [f'node["{tag}"](around:{radius},{lat},{lon});' for tag in QUERY_TAGS]
        + [f'way["{tag}"](around:{radius},{lat},{lon});' for tag in QUERY_TAGS]
    )
    query = f"[out:json][timeout:60];\n(\n    {tag_filters}\n);\nout center tags;"
    
    # Check cache
    cp = _cache_path(query)
    if os.path.exists(cp):
        with open(cp, encoding="utf-8") as f:
            return json.load(f)
    
    # Query with retries across endpoints
    last_error = None
    for attempt in range(max_retries):
        ep = OVERPASS_ENDPOINTS[attempt % len(OVERPASS_ENDPOINTS)]
        try:
            r = requests.post(ep, data={"data": query}, headers=HEADERS, timeout=90)
            r.raise_for_status()
            data = r.json()
            # Cache result
            with open(cp, "w", encoding="utf-8") as f:
                json.dump(data, f)
            return data
        except Exception as e:
            last_error = e
            time.sleep(3 + attempt * 2)
    raise RuntimeError(f"Overpass query failed after {max_retries} retries: {last_error}")


def categorize_elements(data):
    """Extract and categorize OSM elements from Overpass response."""
    counts = {cat: 0 for cat in CATEGORIES}
    total = 0
    
    for element in data.get("elements", []):
        tags = element.get("tags", {})
        # Find the primary value from our query tags
        for tag_key in QUERY_TAGS:
            if tag_key in tags:
                value = tags[tag_key]
                cat = AMENITY_CATEGORIES.get(value)
                if cat:
                    counts[cat] += 1
                    total += 1
                break  # only count each element once
    
    return counts, total


print("Query functions ready.")

In [ ]:
# ── Query amenities for each tract ────────────────────
import math

# Approximate tract area (km2) from radius
TRACT_AREA_KM2 = math.pi * (QUERY_RADIUS / 1000) ** 2

records = []
n_tracts = len(df_tracts)

for i, row in df_tracts.iterrows():
    tract_id = row["tract_id"]
    lat, lon = row["tract_lat"], row["tract_lon"]
    
    if (i + 1) % 25 == 0 or i == 0:
        print(f"  [{i+1}/{n_tracts}] Tract {tract_id} ({lat:.4f}, {lon:.4f})")
    
    try:
        data = query_overpass(lat, lon, QUERY_RADIUS)
        counts, total = categorize_elements(data)
    except Exception as e:
        print(f"  ERROR tract {tract_id}: {e}")
        counts = {cat: 0 for cat in CATEGORIES}
        total = 0
    
    rec = {"tract_id": tract_id, "amenity_count_total": total}
    for cat in CATEGORIES:
        rec[f"amenity_count_{cat}"] = counts[cat]
    
    # Ratios (avoid division by zero)
    if total > 0:
        for cat in CATEGORIES:
            rec[f"amenity_ratio_{cat}"] = round(counts[cat] / total, 4)
    else:
        for cat in CATEGORIES:
            rec[f"amenity_ratio_{cat}"] = 0.0
    
    # Density (amenities per km2)
    rec["amenity_density"] = round(total / TRACT_AREA_KM2, 2)
    
    records.append(rec)
    
    # Rate limiting
    if not os.path.exists(_cache_path("")):
        time.sleep(1)

df_amenities = pd.DataFrame(records)
print(f"\nCompleted: {len(df_amenities)} tracts")
print(f"Mean amenities per tract: {df_amenities['amenity_count_total'].mean():.1f}")
print(f"Tracts with zero amenities: {(df_amenities['amenity_count_total'] == 0).sum()}")

In [ ]:
# ── Save output ───────────────────────────────────────
output_path = "csv/02_amenity_composition.csv"
df_amenities.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}  ({len(df_amenities)} rows x {df_amenities.shape[1]} cols)")

# Show count columns summary
count_cols = [c for c in df_amenities.columns if c.startswith("amenity_count_")]
print("\nAmenity count summary:")
print(df_amenities[count_cols].describe().round(1).to_string())